In [1]:
import re
import math
import random
import numpy as np
import pandas as pd
import nltk
import spacy
import torch

from collections import Counter

from sklearn.feature_extraction.text import*
from sklearn.metrics import*
from sklearn.metrics.pairwise import*

from nltk.stem import*
from nltk.corpus import wordnet as wn
from nltk.wsd import*

from gensim.models import*
from transformers import * 

C:\Users\nikil\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
df = pd.read_csv("data1.csv")

In [3]:
df.head()

,text,sentiment
0,I love this phone and its amazing battery,positive
1,The camera quality is excellent,positive
2,This laptop works very well,positive
3,The service was helpful and friendly,positive
4,I am happy with this product,positive


In [4]:
df.columns
df.iloc[2]

text         This laptop works very well
sentiment                       positive
Name: 2, dtype: str

In [5]:
texts = df["text"].dropna().astype(str).tolist()
labels = df["sentiment"].dropna().astype(str).tolist()

In [6]:
text = "The service was helpful and friendly!"
tokens = re.findall(r"[a-z]+",text.lower())

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

print("token:",tokens)
print("Stems :",[stemmer.stem(w) for w in tokens])
print("Lemmas:",[lemmatizer.lemmatize(w)for w in tokens])

token: ['the', 'service', 'was', 'helpful', 'and', 'friendly']
Stems : ['the', 'servic', 'wa', 'help', 'and', 'friendli']
Lemmas: ['the', 'service', 'wa', 'helpful', 'and', 'friendly']


In [7]:
sentence = [re.findall(r"[a-z]+",t.lower()) for t in texts]

words = Counter(w for s in sentence for w in s)
pairs = Counter()
context = Counter()

for s in sentence :
    for a,b in zip(s,s[1:]):
        pairs[a,b] +=1
        context[a]+=1

V = len(words)
s = sentence[0]
p = 1

for a,b in zip(s,s[1:]):
    p *=(pairs[a,b]+1 / context[a]+V)

print("First Sentence:",texts[2])
print("Probability:",p)
print("Perplexity:",p **(-1/ (len(s)-1)))

First Sentence: This laptop works very well
Probability: 104755182968241.86
Perplexity: 0.00993385423120211


In [8]:
tokens = nltk.word_tokenize(texts[8])
nltk.pos_tag(tokens)

[('The', 'DT'),
 ('battery', 'NN'),
 ('life', 'NN'),
 ('is', 'VBZ'),
 ('great', 'JJ')]

In [9]:


grammar = nltk.CFG.fromstring("""
S -> NP VP
VP -> V NP | VP PP
NP -> DT N | NP PP | 'i'
PP -> P NP
V -> 'saw'
DT -> 'the'
N -> 'man' | 'telescope'
P -> 'with'
""")

parser = nltk.ChartParser(grammar)
trees = list(parser.parse("i saw the man with the telescope".split()))

len(trees), trees

(2,
 [Tree('S', [Tree('NP', ['i']), Tree('VP', [Tree('VP', [Tree('V', ['saw']), Tree('NP', [Tree('DT', ['the']), Tree('N', ['man'])])]), Tree('PP', [Tree('P', ['with']), Tree('NP', [Tree('DT', ['the']), Tree('N', ['telescope'])])])])]),
  Tree('S', [Tree('NP', ['i']), Tree('VP', [Tree('V', ['saw']), Tree('NP', [Tree('NP', [Tree('DT', ['the']), Tree('N', ['man'])]), Tree('PP', [Tree('P', ['with']), Tree('NP', [Tree('DT', ['the']), Tree('N', ['telescope'])])])])])])])

In [10]:
nlp = spacy.load("en_core_web_sm")
doc = nlp("The students are studying machine learning")
for token in doc:
    print(token.text,token.dep_, "->",token.head.text)

The det -> students
students nsubj -> studying
are aux -> studying
studying ROOT -> studying
machine compound -> learning
learning dobj -> studying


In [11]:
wn.synsets("bat")[:5]
wn.synsets("bat")[0].wup_similarity(wn.synsets("ball")[0])

sentence = "He hit the ball with a bat."
sense = lesk(nltk.word_tokenize(sentence),"bat")

sense , sense.definition() if sense else "No sense Found"


(Synset('bat.v.01'), 'strike with, or as if with a baseball bat')

In [12]:
vectorizer = TfidfVectorizer()
matrix = vectorizer.fit_transform(texts)

scores = cosine_similarity(matrix[3],matrix).flatten()

for i in scores.argsort()[::-1][:3]:
    print(round(scores[i],3),texts[i])

1.0 The service was helpful and friendly
0.449 The service was rude and disappointing
0.261 The delivery was late and bad


In [13]:
nlp = spacy.load("en_core_web_sm")
doc = nlp("Nila is a dancer, she conducted a competion for her club.It will be conducted in monday 9.00am")

[e.text for e in doc.ents if e.label_ == "PERSON"]
[c.text for c in doc.noun_chunks]
[t.text for t in doc if t.pos_ == "PRON"]

['she', 'her', 'It']

In [14]:
sentiment = pipeline("sentiment-analysis")
print(sentiment("This phone is terrible!"))
print(sentiment("This phone is ok !"))

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
[transformers] loading configuration file config.json from cache at C:\Users\nikil\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english\snapshots\714eb0fa89d2f80546fda750413ed43d93601a13\config.json
[transformers] Model config DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForSequenceClassification"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "eos_token_id": null,
  "finetuning_task": "sst-2",
  "hidden_dim": 3072,
  "id2label": {
    "0": "NEGATIVE",
    "1": "POSITIVE"
  },
  "initializer_range": 0.02,
  "label2id": {
    "NEGATIVE": 0,
    "POSITIVE": 1
  },
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_l

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] loading configuration file config.json from cache at C:\Users\nikil\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english\snapshots\714eb0fa89d2f80546fda750413ed43d93601a13\config.json
[transformers] Model config DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForSequenceClassification"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "eos_token_id": null,
  "finetuning_task": "sst-2",
  "hidden_dim": 3072,
  "id2label": {
    "0": "NEGATIVE",
    "1": "POSITIVE"
  },
  "initializer_range": 0.02,
  "label2id": {
    "NEGATIVE": 0,
    "POSITIVE": 1
  },
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "output_past": true,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.17.0"

[{'label': 'NEGATIVE', 'score': 0.9996218681335449}]
[{'label': 'POSITIVE', 'score': 0.999535083770752}]


In [15]:
fill_mask = pipeline("fill-mask", model="bert-base-uncased")
fill_mask("The battery is [MASK].", top_k=3)

[transformers] loading configuration file config.json from cache at C:\Users\nikil\.cache\huggingface\hub\models--bert-base-uncased\snapshots\86b5e0934494bd15c9632b12f734a8a67f723594\config.json
[transformers] Model config BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.17.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

[transformers] loading configuratio

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] loading configuration file config.json from cache at C:\Users\nikil\.cache\huggingface\hub\models--bert-base-uncased\snapshots\86b5e0934494bd15c9632b12f734a8a67f723594\config.json
[transformers] Model config BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidde

[{'score': 0.055055566132068634,
  'token': 5338,
  'token_str': 'charged',
  'sequence': 'the battery is charged.'},
 {'score': 0.04081689566373825,
  'token': 2757,
  'token_str': 'dead',
  'sequence': 'the battery is dead.'},
 {'score': 0.03557557612657547,
  'token': 2489,
  'token_str': 'free',
  'sequence': 'the battery is free.'}]